# Drug response in bulk RNA-seq, from a public count matrix

**Data kit lab** `airway-dexamethasone-fold-change` · Discovery · about 30 minutes

**The question.** By how much does dexamethasone raise the expression of CRISPLD2, DUSP1, KLF15, PER1, TSC22D3 and FKBP5 in primary human airway smooth muscle cells?

**What you do.**

1. Read the sample sheet of GEO series GSE52778 and pick the two groups.
2. Read NCBI's raw count matrix and find the six genes.
3. Normalise each sample to counts per million.
4. Compute the fold change of the group means, on the log2 scale.
5. Compare with Table 1 of the paper.

**The data.** Source: NCBI Gene Expression Omnibus, series GSE52778, submitted by Himes and others. NCBI states it places no restrictions on the use or distribution of its molecular data; some submitters may claim rights, so credit them. Nothing is copied into this notebook: the first cell downloads the files from NCBI (the count matrix is about 0.7 MB; the NCBI Gene table about 5 MB).

**Important.** The paper's numbers come from its own pipeline (Cufflinks and Cuffdiff on the hg19 genome). This lab uses the raw counts NCBI generated later from the same reads on GRCh38.p13. Same experiment, different route: expect the same direction and similar sizes, not the same decimals.

**The paper.** Himes BE and others. RNA-Seq transcriptome profiling identifies CRISPLD2 as a glucocorticoid responsive gene that modulates cytokine function in airway smooth muscle cells. PLoS One 2014. https://doi.org/10.1371/journal.pone.0099625 . Keep Table 1 open.

**You need.** Python 3.10 or later with `pandas` and `numpy`.

In [ ]:
# datakit:setup (generated from lab.yaml and expected.json by scripts/datakit/harness.py sync; do not edit by hand)
import hashlib, json, os, pathlib, urllib.request

LAB = 'airway-dexamethasone-fold-change'
FILES = json.loads(r'''{
 "counts": {
  "url": "https://ftp.ncbi.nlm.nih.gov/geo/rnaseq_counts/GSE52nnn/GSE52778_raw_counts_GRCh38.p13_NCBI.tsv.gz",
  "name": "counts.gz",
  "sha256": "8f5d0adafcb5d943338e196b99aa87dd8ca2515927450627b7443328a3956603",
  "bytes": 740924
 },
 "genes": {
  "url": "https://ftp.ncbi.nlm.nih.gov/gene/DATA/GENE_INFO/Mammalia/Homo_sapiens.gene_info.gz",
  "name": "genes.gz",
  "sha256": "0403f76eb4fe766fa697150a532a74ed47639af9a417a42af7618f468eee1bcb",
  "bytes": 5203032
 },
 "series": {
  "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE52nnn/GSE52778/matrix/GSE52778_series_matrix.txt.gz",
  "name": "series.gz",
  "sha256": "67db46e962f795b9dd61fa6b4279f2678c6b2b9bf1df035fbc22fcf81b5e879e",
  "bytes": 3396
 }
}''')
PUBLISHED = json.loads(r'''{
 "log2fc_crispld2": {
  "label": "log2 fold change Dex vs untreated, CRISPLD2",
  "value": 2.7,
  "tolerance": {
   "abs": 0.005
  }
 },
 "up_crispld2": {
  "label": "Direction of change with Dex, CRISPLD2",
  "value": "up",
  "tolerance": {
   "exact": true
  }
 },
 "log2fc_dusp1": {
  "label": "log2 fold change Dex vs untreated, DUSP1",
  "value": 2.99,
  "tolerance": {
   "abs": 0.005
  }
 },
 "up_dusp1": {
  "label": "Direction of change with Dex, DUSP1",
  "value": "up",
  "tolerance": {
   "exact": true
  }
 },
 "log2fc_klf15": {
  "label": "log2 fold change Dex vs untreated, KLF15",
  "value": 4.58,
  "tolerance": {
   "abs": 0.005
  }
 },
 "up_klf15": {
  "label": "Direction of change with Dex, KLF15",
  "value": "up",
  "tolerance": {
   "exact": true
  }
 },
 "log2fc_per1": {
  "label": "log2 fold change Dex vs untreated, PER1",
  "value": 3.2,
  "tolerance": {
   "abs": 0.005
  }
 },
 "up_per1": {
  "label": "Direction of change with Dex, PER1",
  "value": "up",
  "tolerance": {
   "exact": true
  }
 },
 "log2fc_tsc22d3": {
  "label": "log2 fold change Dex vs untreated, TSC22D3",
  "value": 3.27,
  "tolerance": {
   "abs": 0.005
  }
 },
 "up_tsc22d3": {
  "label": "Direction of change with Dex, TSC22D3",
  "value": "up",
  "tolerance": {
   "exact": true
  }
 },
 "log2fc_fkbp5": {
  "label": "log2 fold change Dex vs untreated, FKBP5",
  "value": 3.95,
  "tolerance": {
   "abs": 0.005
  }
 },
 "up_fkbp5": {
  "label": "Direction of change with Dex, FKBP5",
  "value": "up",
  "tolerance": {
   "exact": true
  }
 }
}''')
PUBLISHED_FROM = 'Europe PMC full-text record PMC4057123 (XML, no page numbers): Table 1, Top (Q-value <1E-10) differentially expressed genes, column Ln[ Fold Change ], rows CRISPLD2, DUSP1, KLF15, PER1, TSC22D3, FKBP5 | https://www.ebi.ac.uk/europepmc/webservices/rest/PMC4057123/fullTextXML'
DATA_DIR = pathlib.Path(os.environ.get('DATAKIT_DATA', 'datakit-data/' + LAB))


def data_file(file_id):
    """Path of one dataset file. Downloads it from its source on first use and checks the checksum."""
    f = FILES[file_id]
    path = DATA_DIR / f['name']
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print('downloading', f['url'])
        urllib.request.urlretrieve(f['url'], path)
    if f['sha256']:
        got = hashlib.sha256(path.read_bytes()).hexdigest()
        if got != f['sha256']:
            print('WARNING:', path.name, 'differs from the copy this lab was checked against (the source file changed)')
    return path


def within(computed, check):
    tol, pub = check['tolerance'], check['value']
    if 'exact' in tol:
        return computed == pub
    limit = tol['abs'] if 'abs' in tol else tol['rel'] * abs(pub)
    return abs(computed - pub) <= limit + 1e-12


def report(results):
    """Print computed against published values and save the computed values for the harness."""
    print('Published values:', PUBLISHED_FROM)
    print(f"{'check':<44}{'published':>12}{'computed':>14}  result")
    for key, check in PUBLISHED.items():
        got = results.get(key)
        shown = '(missing)' if got is None else (f'{got:.4f}' if isinstance(got, float) else str(got))
        verdict = 'missing' if got is None else ('match' if within(got, check) else 'DIFFERENT')
        print(f"{check['label'][:43]:<44}{str(check['value']):>12}{shown:>14}  {verdict}")
    out = pathlib.Path(os.environ.get('DATAKIT_RESULTS', 'results.json'))
    out.write_text(json.dumps(results, indent=1))
    print('saved', out)


## 1. The sample sheet

The series matrix holds one column per sample. Its `!Sample_characteristics_ch1` lines carry the treatment and the cell line (one cell line per donor). Build a small table from it.

In [ ]:
import gzip

import numpy as np
import pandas as pd

with gzip.open(data_file("series"), "rt", encoding="utf-8") as fh:
    lines = [line.rstrip("\n").split("\t") for line in fh if line.startswith("!Sample_")]

def field(name):
    return [v.strip('"') for v in next(l for l in lines if l[0] == name)[1:]]

sheet = pd.DataFrame({"gsm": field("!Sample_geo_accession"), "title": field("!Sample_title")})
for l in lines:
    if l[0] == "!Sample_characteristics_ch1":
        key = l[1].strip('"').split(":")[0]
        sheet[key] = [v.strip('"').split(": ", 1)[1] for v in l[1:]]
print(len(sheet), "samples")
sheet[["gsm", "title", "cell line", "treatment", "ercc_mix"]]

The series has 16 samples: four cell lines, each under four treatments. The paper compares dexamethasone alone with untreated cells. Pick those two groups by the treatment field, not by column position.

In [ ]:
groups = {"untreated": "Untreated", "dex": "Dexamethasone"}
chosen = sheet[sheet["treatment"].isin(groups.values())]
for key, label in groups.items():
    n = (chosen["treatment"] == label).sum()
    assert n == 4, f"{label}: {n} samples"
# one sample of each group per cell line
assert (chosen.groupby("cell line")["treatment"].nunique() == 2).all()
untreated = chosen.loc[chosen["treatment"] == "Untreated", "gsm"].tolist()
dex = chosen.loc[chosen["treatment"] == "Dexamethasone", "gsm"].tolist()
print("untreated:", untreated)
print("dexamethasone:", dex)

## 2. The count matrix

Rows are NCBI GeneIDs, columns are GSM ids, values are raw read counts. NCBI generated it on GRCh38.p13.

In [ ]:
counts = pd.read_csv(data_file("counts"), sep="\t", index_col=0, compression="gzip")
print(counts.shape, "genes x samples")
assert set(untreated + dex) <= set(counts.columns)
counts = counts[untreated + dex]
print("reads assigned to genes, millions:")
print((counts.sum() / 1e6).round(1).to_string())

## 3. From gene id to gene name

The matrix has no gene names. NCBI Gene publishes a table of every human GeneID with its symbol. Read only the three columns you need. (GEO also offers its own annotation table for these matrices; its download link answered a script with a browser check when this lab was written, so the lab uses NCBI Gene's table. It is the current release, so check that each symbol gives exactly one id that is in the matrix.)

In [ ]:
genes = pd.read_csv(data_file("genes"), sep="\t", usecols=["#tax_id", "GeneID", "Symbol"], compression="gzip")
genes = genes[genes["#tax_id"] == 9606]
TARGETS = ["CRISPLD2", "DUSP1", "KLF15", "PER1", "TSC22D3", "FKBP5"]
gene_id = {}
for sym in TARGETS:
    ids = genes.loc[genes["Symbol"] == sym, "GeneID"].tolist()
    assert len(ids) == 1, f"{sym}: {ids}"
    assert ids[0] in counts.index, f"{sym} (GeneID {ids[0]}) is not in the matrix"
    gene_id[sym] = ids[0]
print(gene_id)
counts.loc[list(gene_id.values())].rename(index={v: k for k, v in gene_id.items()})

## 4. Normalise, then take the ratio of the group means

Table 1 prints, for each gene, the mean FPKM in the control group, the mean FPKM in the dexamethasone group, and the logarithm of their ratio. FPKM divides reads by the gene's length and by the sample's millions of reads. For one gene the length is the same in both groups, so it cancels in the ratio. The count-matrix version of the same quantity is therefore the ratio of mean counts per million (CPM).

Steps: CPM per sample (divide by that sample's total over all genes in the matrix), mean per group, ratio, log2. No filter, no pseudocount.

In [ ]:
cpm = counts / counts.sum() * 1e6
mean_untreated = cpm[untreated].mean(axis=1)
mean_dex = cpm[dex].mean(axis=1)
rows = []
for sym, gid in gene_id.items():
    assert mean_untreated[gid] > 0
    rows.append({"gene": sym, "mean_cpm_untreated": mean_untreated[gid], "mean_cpm_dex": mean_dex[gid],
                 "log2_fold_change": float(np.log2(mean_dex[gid] / mean_untreated[gid]))})
fc = pd.DataFrame(rows).set_index("gene")
fc.round(3)

## 5. How much do the donors differ?

The group-mean ratio hides the spread between donors. Each cell line has one untreated and one dexamethasone sample, so a ratio can be taken within each donor too. This is shown, not checked: it is a different quantity from Table 1's.

In [ ]:
line_of = dict(zip(chosen["gsm"], chosen["cell line"]))
per_donor = {}
for u in untreated:
    d = next(x for x in dex if line_of[x] == line_of[u])
    per_donor[line_of[u]] = np.log2(cpm.loc[list(gene_id.values()), d] / cpm.loc[list(gene_id.values()), u]).values
pd.DataFrame(per_donor, index=list(gene_id)).round(2)

## 6. Check against the paper

`PUBLISHED` holds the values of Table 1, column "Ln[ Fold Change ]", copied by code from the full text. The header says Ln, but the numbers are log2 of the two mean FPKM columns (the extractor tested this on all 17 rows). `report` prints your value beside each one. The tolerance is half a unit of the last printed digit: 0.005.

In [ ]:
RESULTS = {}
for sym in TARGETS:
    value = fc.loc[sym, "log2_fold_change"]
    RESULTS[f"log2fc_{sym.lower()}"] = value
    RESULTS[f"up_{sym.lower()}"] = "up" if value > 0 else "down"
for sym in TARGETS:
    pub = PUBLISHED[f"log2fc_{sym.lower()}"]["value"]
    print(f"{sym:<9} computed {fc.loc[sym, 'log2_fold_change']:6.2f}  printed {pub:5.2f}  difference {fc.loc[sym, 'log2_fold_change'] - pub:+.2f}")
report(RESULTS)

## What the result says

Read the direction rows first: does dexamethasone raise all six genes in NCBI's counts, as in the paper? Then read the differences. They come from two different pipelines run on the same reads: another genome build (hg19 against GRCh38.p13), another aligner and annotation, and gene counts against Cufflinks FPKM with bias correction. The paper's mean FPKM is also Cuffdiff's own estimate per condition. A difference in the second decimal is not a failure of the biology; it is the size of a pipeline choice.

## Try next

1. Add the albuterol samples to the untreated group by mistake (use all samples whose treatment is not dexamethasone). How far do the fold changes move?
2. Compute the mean of the four per-donor log2 ratios instead of the log2 of the ratio of means. Which genes change most, and why are they the low-expression ones (KLF15, PER1)?
3. GEO also hosts NCBI's FPKM matrix for this series and the authors' own Cuffdiff output (see the download page). Which of them would you expect to be closest to Table 1?
4. Run a differential expression tool (for example PyDESeq2) on the 8 samples with donor as a covariate. Its log2 fold change is a model estimate, not a ratio of means: compare it with both.